# M04 — Linear SVM và RBF SVM

**Chủ phần:** Hưng — Task T16, T20, T24, T28

**Ngày:** 2026-11-23  
**Dataset:** CDC Diabetes Health Indicators (UCI #891) — `data/splits/train.csv` (niêm phong `test.csv` tới M06)  
**Metric chính:** `average_precision` (AP) — phù hợp với dữ liệu mất cân bằng (~15.3% positive)  
**CV:** 5-fold StratifiedKFold, seed 42, chỉ trên tập train 80%

---

## Nội dung notebook

1. Lý thuyết SVM — Linear và RBF  
2. Setup và load dữ liệu  
3. Tổng quan dataset  
4. Linear SVM — Baseline CV  
5. Linear SVM — Hyperparameter tuning (GridSearch)  
6. RBF SVM — Baseline CV  
7. RBF SVM — Hyperparameter tuning (GridSearch)  
8. So sánh Linear vs RBF  
9. Fit best pipelines và xuất artifacts  
10. Validation artifacts và nhận xét

## 1. Lý thuyết — T16: Nghiên cứu Linear SVM, Kernel SVM và margin

### 1.1 Support Vector Machine tuyến tính

SVM tìm siêu phẳng (hyperplane) $w^\top x + b = 0$ phân tách hai lớp với **biên (margin)** lớn nhất.

**Bài toán tối ưu (Soft-margin SVM — C-SVM):**
$$\min_{w, b, \xi} \frac{1}{2}\|w\|^2 + C \sum_{i=1}^n \xi_i$$
$$\text{s.t.} \quad y_i(w^\top x_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

- $C > 0$: hệ số phạt vi phạm margin. **C lớn** → ít bỏ qua lỗi → dễ overfit; **C nhỏ** → chấp nhận lỗi nhiều → regularization mạnh hơn.
- $\xi_i$: slack variable — cho phép điểm nằm sai phía.
- **Hinge loss**: $\ell(y, f(x)) = \max(0, 1 - y \cdot f(x))$

**`LinearSVC`** (thư viện LIBLINEAR): tối ưu trực tiếp bài toán primal/dual tuyến tính, **không xây dựng kernel matrix** → $O(n \cdot d)$ thay vì $O(n^2)$ → phù hợp n=183k.

### 1.2 Kernel SVM — RBF

Kernel trick ánh xạ dữ liệu sang không gian chiều cao hơn bằng hàm kernel $K(x_i, x_j)$, **không tính tường minh** vector ánh xạ:

$$K_{\text{RBF}}(x_i, x_j) = \exp\left(-\gamma \|x_i - x_j\|^2\right)$$

- $\gamma > 0$: quy định "bán kính ảnh hưởng" của mỗi điểm dữ liệu.  
  - **$\gamma$ lớn** → ảnh hưởng cục bộ → dễ overfit; **$\gamma$ nhỏ** → ảnh hưởng rộng → underfitting.
  - `gamma='scale'`: $\gamma = 1/(d \cdot \text{Var}(X))$ — mặc định sklearn ≥ 0.22.
  - `gamma='auto'`: $\gamma = 1/d$.
- **Chi phí tính toán**: $O(n^2)$ để xây dựng kernel matrix → chậm hơn LinearSVC trên n=183k. Chỉ "thử" RBF như một thí nghiệm phụ theo roadmap.

### 1.3 Linear vs RBF — khi nào dùng cái nào?

| Tiêu chí | LinearSVC | SVC (RBF) |
|---|---|---|
| Không gian quyết định | Tuyến tính | Phi tuyến |
| Chi phí train | $O(n \cdot d)$ | $O(n^2 \cdot d)$ |
| Tham số | C | C, gamma |
| Phù hợp | n lớn, d lớn, quan hệ gần tuyến tính | n vừa, quan hệ phi tuyến |
| Khả năng giải thích | Weight vector trực tiếp | Dựa trên support vectors |

### 1.4 Score và ROC-AUC/AP

Cả `LinearSVC` và `SVC(probability=False)` đều sử dụng `decision_function` thay vì `predict_proba`. Giá trị `decision_function` là khoảng cách có dấu từ hyperplane — không phải xác suất nhưng là **ranking score** hợp lệ cho ROC-AUC và Average Precision. Hàm `get_continuous_scores()` trong `src/evaluation.py` xử lý tự động.

### 1.5 class_weight và dữ liệu mất cân bằng

Dataset có ~15.3% positive (tiểu đường/tiền tiểu đường). `class_weight='balanced'` điều chỉnh trọng số lớp tỷ lệ nghịch với tần suất:
$$w_i = \frac{n}{k \cdot n_i}$$
Giúp model không thiên về lớp đa số (y=0).

## 2. Setup và load dữ liệu — T20 (phần 1)

In [ ]:
"""Setup cell — imports, path configuration, reproducibility."""
import sys
from pathlib import Path

# Ensure project root is on sys.path (works whether run from notebooks/ or root)
PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
)

from src.data_split import load_config, load_train_data
from src.evaluation import (
    build_metrics_record,
    evaluate_cv,
    get_continuous_scores,
    save_metrics_json,
    validate_metrics_record,
)
from src.models.svm import (
    LINEAR_PARAM_GRID,
    RBF_PARAM_GRID,
    build_linear_svm_pipeline,
    build_rbf_svm_pipeline,
    run_baseline_cv,
    run_grid_search,
    run_svm_experiment,
    save_pipeline,
)
from src.preprocessing import get_cv_splitter

warnings.filterwarnings("ignore", category=UserWarning)

print(f"Python  : {sys.version.split()[0]}")
print(f"sklearn : {sklearn.__version__}")
print(f"Project : {PROJECT_ROOT}")

CONFIG = load_config()
ARTIFACTS_DIR = PROJECT_ROOT / CONFIG["paths"]["artifacts"]
METRICS_DIR   = PROJECT_ROOT / CONFIG["paths"]["metrics"]
FIGURES_DIR   = PROJECT_ROOT / CONFIG["paths"]["figures"]
for d in (ARTIFACTS_DIR, METRICS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Config loaded — primary metric: {CONFIG['training']['primary_metric']}")

In [ ]:
"""Load 80% training split — test set remains sealed until M06."""
X_train, y_train = load_train_data(config=CONFIG)
print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}  |  dtype: {y_train.dtype}")
print(f"Positive rate: {y_train.mean():.4f} ({y_train.sum():,} / {len(y_train):,})")

## 3. Tổng quan dataset

In [ ]:
"""Dataset overview — class distribution and feature types."""
print("=== Class distribution (y_train) ===")
counts = y_train.value_counts().sort_index()
for label, cnt in counts.items():
    print(f"  y={label}: {cnt:>7,}  ({cnt/len(y_train)*100:.2f}%)")

print("\n=== Feature dtypes ===")
print(X_train.dtypes.value_counts().to_string())

print("\n=== Feature summary (sample) ===")
print(X_train[["BMI", "GenHlth", "Age", "HighBP"]].describe().loc[["min", "max", "mean"]].round(2))

# Class imbalance bar chart
fig, ax = plt.subplots(figsize=(4, 3))
counts.plot.bar(ax=ax, color=["steelblue", "tomato"], edgecolor="black", width=0.5)
ax.set_xticklabels(["0 (No diabetes)", "1 (Diabetes)"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Class Distribution — Train Set")
for i, v in enumerate(counts):
    ax.text(i, v + 500, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "svm_class_distribution.png", dpi=120)
plt.show()

## 4. Linear SVM — Baseline CV (T20)

Baseline dùng `LinearSVC` với `C=1.0`, `class_weight='balanced'`, `max_iter=5000`.
Pipeline: `StandardScaler → LinearSVC` — built bởi `build_linear_svm_pipeline()`.
CV: 5-fold StratifiedKFold, scoring = tất cả 6 metrics, AP là metric chính.

In [ ]:
"""Linear SVM baseline — build pipeline and run 5-fold CV."""
linear_baseline_pipe = build_linear_svm_pipeline(X_train)
print("Pipeline:")
print(linear_baseline_pipe)

print("\nRunning 5-fold StratifiedKFold CV (baseline)...")
linear_baseline_result = run_baseline_cv(
    linear_baseline_pipe, X_train, y_train, config=CONFIG, n_jobs=-1
)

val_cv_bl = linear_baseline_result["validation_cv"]
print(f"\n{'Metric':<25} {'Mean':>8} {'Std':>8}")
print("-" * 43)
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]:
    m = val_cv_bl[f"{metric}_mean"]
    s = val_cv_bl[f"{metric}_std"]
    flag = " ← PRIMARY" if metric == "average_precision" else ""
    print(f"  {metric:<23} {m:>8.4f} {s:>8.4f}{flag}")
print(f"\nBaseline CV time: {linear_baseline_result['training_time_seconds']:.1f}s")

LINEAR_BASELINE_AP = val_cv_bl["average_precision_mean"]
print(f"\nBaseline AP (mean): {LINEAR_BASELINE_AP:.4f}")

## 5. Linear SVM — Hyperparameter Tuning (T24)

GridSearchCV trên training folds, scoring=`average_precision`.

**Search space:**
- `model__C`: [0.001, 0.01, 0.1, 1, 10]
- `model__class_weight`: ['balanced', None]

Tổng: 10 điểm × 5 folds = 50 fits.

In [ ]:
"""Linear SVM GridSearchCV."""
linear_pipe_for_gs = build_linear_svm_pipeline(X_train)
print(f"Search grid: {LINEAR_PARAM_GRID}")
print(f"Total fits: {5 * sum(len(v) for v in LINEAR_PARAM_GRID.values())} (approx)")

linear_gs = run_grid_search(
    linear_pipe_for_gs, X_train, y_train, LINEAR_PARAM_GRID,
    config=CONFIG, n_jobs=-1, verbose=1,
)

print(f"\nBest AP (CV): {linear_gs.best_score_:.4f}")
print(f"Best params : {linear_gs.best_params_}")
print(f"Search time : {linear_gs.search_time_seconds_:.1f}s")

In [ ]:
"""Analyse GridSearch results for Linear SVM."""
cv_results_df = pd.DataFrame(linear_gs.cv_results_)
print(cv_results_df[
    ["param_model__C", "param_model__class_weight",
     "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score").to_string(index=False))

# Pivot for heatmap-style view
pivot = cv_results_df.pivot_table(
    index="param_model__C",
    columns="param_model__class_weight",
    values="mean_test_score",
    aggfunc="mean",
).sort_index()
print("\nAP mean by (C, class_weight):")
print(pivot.to_string(float_format="{:.4f}".format))

In [ ]:
"""Plot C vs AP for different class_weight values."""
cv_df = pd.DataFrame(linear_gs.cv_results_)
fig, ax = plt.subplots(figsize=(7, 4))
for cw in ["balanced", None]:
    subset = cv_df[cv_df["param_model__class_weight"] == cw].copy()
    subset = subset.sort_values("param_model__C")
    label = f"class_weight={cw!r}"
    ax.plot(
        subset["param_model__C"],
        subset["mean_test_score"],
        marker="o", label=label,
    )
    ax.fill_between(
        subset["param_model__C"],
        subset["mean_test_score"] - subset["std_test_score"],
        subset["mean_test_score"] + subset["std_test_score"],
        alpha=0.15,
    )
ax.set_xscale("log")
ax.set_xlabel("C (regularization)")
ax.set_ylabel("Average Precision (CV mean)")
ax.set_title("LinearSVC — AP vs C (5-fold CV)")
ax.legend()
ax.axhline(LINEAR_BASELINE_AP, color="gray", linestyle="--", label="baseline")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "svm_linear_gridsearch.png", dpi=120)
plt.show()

In [ ]:
"""Run full 6-metric CV on best Linear SVM pipeline."""
print(f"Best LinearSVC params: {linear_gs.best_params_}")
print("\nRunning full 6-metric CV on best pipeline...")

linear_best_cv = run_baseline_cv(
    linear_gs.best_estimator_, X_train, y_train, config=CONFIG, n_jobs=-1
)
linear_val_cv = linear_best_cv["validation_cv"]
linear_train_time = linear_best_cv["training_time_seconds"]

print(f"\n{'Metric':<25} {'Mean':>8} {'Std':>8}")
print("-" * 43)
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]:
    m = linear_val_cv[f"{metric}_mean"]
    s = linear_val_cv[f"{metric}_std"]
    flag = " ← PRIMARY" if metric == "average_precision" else ""
    print(f"  {metric:<23} {m:>8.4f} {s:>8.4f}{flag}")
print(f"\nFull CV time: {linear_train_time:.1f}s")

## 6. RBF SVM — Baseline CV (T20)

Baseline dùng `SVC(kernel='rbf', probability=False)` với `C=1.0`, `gamma='scale'`, `class_weight='balanced'`.
Pipeline: `StandardScaler → SVC(rbf)` — built bởi `build_rbf_svm_pipeline()`.

> **Note:** RBF SVC trên 183k rows chậm hơn đáng kể so với LinearSVC. Đây là thí nghiệm phụ theo roadmap.

In [ ]:
"""RBF SVM baseline — build pipeline and run 5-fold CV."""
rbf_baseline_pipe = build_rbf_svm_pipeline(X_train)
print("Pipeline:")
print(rbf_baseline_pipe)

print("\nRunning 5-fold StratifiedKFold CV (baseline) — may take a few minutes on 183k rows...")
t0 = time.perf_counter()
rbf_baseline_result = run_baseline_cv(
    rbf_baseline_pipe, X_train, y_train, config=CONFIG, n_jobs=-1
)
print(f"Completed in {time.perf_counter()-t0:.1f}s")

val_cv_rbf_bl = rbf_baseline_result["validation_cv"]
print(f"\n{'Metric':<25} {'Mean':>8} {'Std':>8}")
print("-" * 43)
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]:
    m = val_cv_rbf_bl[f"{metric}_mean"]
    s = val_cv_rbf_bl[f"{metric}_std"]
    flag = " ← PRIMARY" if metric == "average_precision" else ""
    print(f"  {metric:<23} {m:>8.4f} {s:>8.4f}{flag}")
print(f"\nBaseline CV time: {rbf_baseline_result['training_time_seconds']:.1f}s")

RBF_BASELINE_AP = val_cv_rbf_bl["average_precision_mean"]
print(f"\nRBF Baseline AP (mean): {RBF_BASELINE_AP:.4f}")

## 7. RBF SVM — Hyperparameter Tuning (T24)

GridSearchCV, scoring=`average_precision`.

**Search space:**
- `model__C`: [0.1, 1, 10]
- `model__gamma`: ['scale', 'auto', 0.01]
- `model__class_weight`: ['balanced'] — cố định vì dữ liệu mất cân bằng

Tổng: 9 điểm × 5 folds = 45 fits.

In [ ]:
"""RBF SVM GridSearchCV."""
rbf_pipe_for_gs = build_rbf_svm_pipeline(X_train)
print(f"Search grid: {RBF_PARAM_GRID}")
print("Running GridSearchCV (45 fits) — may take a while...")

rbf_gs = run_grid_search(
    rbf_pipe_for_gs, X_train, y_train, RBF_PARAM_GRID,
    config=CONFIG, n_jobs=-1, verbose=1,
)

print(f"\nBest AP (CV): {rbf_gs.best_score_:.4f}")
print(f"Best params : {rbf_gs.best_params_}")
print(f"Search time : {rbf_gs.search_time_seconds_:.1f}s")

In [ ]:
"""Analyse GridSearch results for RBF SVM."""
rbf_cv_df = pd.DataFrame(rbf_gs.cv_results_)
print(rbf_cv_df[
    ["param_model__C", "param_model__gamma",
     "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score").to_string(index=False))

pivot_rbf = rbf_cv_df.pivot_table(
    index="param_model__C",
    columns="param_model__gamma",
    values="mean_test_score",
    aggfunc="mean",
).sort_index()
print("\nAP mean by (C, gamma):")
print(pivot_rbf.to_string(float_format="{:.4f}".format))

In [ ]:
"""Run full 6-metric CV on best RBF SVM pipeline."""
print(f"Best RBF SVC params: {rbf_gs.best_params_}")
print("\nRunning full 6-metric CV on best RBF pipeline...")

rbf_best_cv = run_baseline_cv(
    rbf_gs.best_estimator_, X_train, y_train, config=CONFIG, n_jobs=-1
)
rbf_val_cv = rbf_best_cv["validation_cv"]
rbf_train_time = rbf_best_cv["training_time_seconds"]

print(f"\n{'Metric':<25} {'Mean':>8} {'Std':>8}")
print("-" * 43)
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]:
    m = rbf_val_cv[f"{metric}_mean"]
    s = rbf_val_cv[f"{metric}_std"]
    flag = " ← PRIMARY" if metric == "average_precision" else ""
    print(f"  {metric:<23} {m:>8.4f} {s:>8.4f}{flag}")
print(f"\nFull CV time: {rbf_train_time:.1f}s")

## 8. So sánh Linear SVM vs RBF SVM (validation CV)

In [ ]:
"""Summary comparison table: Baseline vs Best (Linear and RBF)."""
metrics_order = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
rows = []
for label, val_cv in [
    ("LinearSVM Baseline", val_cv_bl),
    ("LinearSVM Tuned",    linear_val_cv),
    ("RBF SVM Baseline",  val_cv_rbf_bl),
    ("RBF SVM Tuned",     rbf_val_cv),
]:
    row = {"Model": label}
    for m in metrics_order:
        row[m] = f"{val_cv[f'{m}_mean']:.4f} ± {val_cv[f'{m}_std']:.4f}"
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("Model")
print(summary_df.to_string())

In [ ]:
"""Bar chart: AP mean comparison (baseline vs tuned, linear vs rbf)."""
models_labels = ["Linear\nBaseline", "Linear\nTuned", "RBF\nBaseline", "RBF\nTuned"]
ap_means = [
    val_cv_bl["average_precision_mean"],
    linear_val_cv["average_precision_mean"],
    val_cv_rbf_bl["average_precision_mean"],
    rbf_val_cv["average_precision_mean"],
]
ap_stds = [
    val_cv_bl["average_precision_std"],
    linear_val_cv["average_precision_std"],
    val_cv_rbf_bl["average_precision_std"],
    rbf_val_cv["average_precision_std"],
]
colors = ["steelblue", "navy", "tomato", "darkred"]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(models_labels, ap_means, yerr=ap_stds, capsize=5, color=colors,
              edgecolor="black", alpha=0.85, width=0.55)
ax.set_ylabel("Average Precision (mean ± std)")
ax.set_title("SVM — AP Comparison (5-fold CV)")
ax.set_ylim(0, min(1.0, max(ap_means) * 1.25))
for bar, m, s in zip(bars, ap_means, ap_stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.005,
            f"{m:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "svm_ap_comparison.png", dpi=120)
plt.show()

## 9. Fit best pipelines và xuất artifacts (T28)

Fit toàn bộ X_train (best params từ GridSearch) → lưu `.joblib` và `.json`.

In [ ]:
"""Save Linear SVM — artifacts/linear_svm.joblib and results/metrics/linear_svm.json."""
linear_artifact = ARTIFACTS_DIR / "linear_svm.joblib"
linear_metrics_path = METRICS_DIR / "linear_svm.json"

# Pipeline is already refitted on full X_train by GridSearchCV (refit=True)
save_pipeline(linear_gs.best_estimator_, linear_artifact)
print(f"Linear SVM pipeline saved → {linear_artifact}")

linear_record = build_metrics_record(
    model_name="linear_svm",
    validation_cv=linear_val_cv,
    best_params=linear_gs.best_params_,
    training_time_seconds=linear_train_time,
    baseline_ap_mean=LINEAR_BASELINE_AP,
    pipeline_path="artifacts/linear_svm.joblib",
    score_method="decision_function",
    config=CONFIG,
)
save_metrics_json(linear_record, linear_metrics_path, config=CONFIG)
print(f"Linear SVM metrics saved → {linear_metrics_path}")

print("\nLinear SVM metrics record:")
print(json.dumps(linear_record, indent=2, ensure_ascii=False))

In [ ]:
"""Save RBF SVM — artifacts/rbf_svm.joblib and results/metrics/rbf_svm.json."""
rbf_artifact = ARTIFACTS_DIR / "rbf_svm.joblib"
rbf_metrics_path = METRICS_DIR / "rbf_svm.json"

save_pipeline(rbf_gs.best_estimator_, rbf_artifact)
print(f"RBF SVM pipeline saved → {rbf_artifact}")

rbf_record = build_metrics_record(
    model_name="rbf_svm",
    validation_cv=rbf_val_cv,
    best_params=rbf_gs.best_params_,
    training_time_seconds=rbf_train_time,
    baseline_ap_mean=RBF_BASELINE_AP,
    pipeline_path="artifacts/rbf_svm.joblib",
    score_method="decision_function",
    config=CONFIG,
)
save_metrics_json(rbf_record, rbf_metrics_path, config=CONFIG)
print(f"RBF SVM metrics saved → {rbf_metrics_path}")

print("\nRBF SVM metrics record:")
print(json.dumps(rbf_record, indent=2, ensure_ascii=False))

## 10. Validation artifacts và nhận xét

In [ ]:
"""Validate that both metrics.json files pass the shared schema check."""
from src.evaluation import validate_metrics_record

for path in [linear_metrics_path, rbf_metrics_path]:
    record = json.loads(path.read_text(encoding="utf-8"))
    validate_metrics_record(record, config=CONFIG)
    print(f"✓ {path.name} — schema valid")

print("\nAll metrics.json files pass schema validation.")

In [ ]:
"""Verify saved pipelines can load and produce predictions."""
from src.models.svm import load_pipeline

for name, path in [("linear_svm", linear_artifact), ("rbf_svm", rbf_artifact)]:
    pipe = load_pipeline(path)
    sample = X_train.iloc[:5]
    preds = pipe.predict(sample)
    scores = pipe.decision_function(sample)
    print(f"✓ {name}: predict={preds.tolist()}, decision_function shape={scores.shape}")

print("\nAll pipeline artifacts load and predict correctly.")

In [ ]:
"""Precision-Recall curve on training set (CV out-of-fold scores) for final pipelines."""
cv = get_cv_splitter(CONFIG)

fig, ax = plt.subplots(figsize=(7, 5))
for name, pipe, color in [
    ("LinearSVM (best)", linear_gs.best_estimator_, "navy"),
    ("RBF SVM (best)",   rbf_gs.best_estimator_,   "darkred"),
]:
    oof_scores = cross_val_predict(
        pipe, X_train, y_train, cv=cv, method="decision_function", n_jobs=-1
    )
    ap = average_precision_score(y_train, oof_scores)
    PrecisionRecallDisplay.from_predictions(
        y_train, oof_scores, name=f"{name} (AP={ap:.3f})", ax=ax, color=color
    )

prevalence = y_train.mean()
ax.axhline(prevalence, color="gray", linestyle="--", label=f"Prevalence={prevalence:.3f}")
ax.set_title("Precision-Recall Curve (Out-of-Fold CV, Train Set)")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "svm_pr_curve.png", dpi=120)
plt.show()

In [ ]:
"""Summary and handoff notes for Hưng → M05 (Hưng) → M06 (Hoàng)."""
print("=" * 60)
print("M04 SVM — Bàn giao T28")
print("=" * 60)
print(f"  Linear SVM best params : {linear_gs.best_params_}")
print(f"  Linear SVM AP (CV)     : {linear_val_cv['average_precision_mean']:.4f} "
      f"± {linear_val_cv['average_precision_std']:.4f}")
print(f"  RBF SVM best params    : {rbf_gs.best_params_}")
print(f"  RBF SVM AP (CV)        : {rbf_val_cv['average_precision_mean']:.4f} "
      f"± {rbf_val_cv['average_precision_std']:.4f}")
print()
print("Artifacts:")
print(f"  {linear_artifact}")
print(f"  {rbf_artifact}")
print("Metrics:")
print(f"  {linear_metrics_path}")
print(f"  {rbf_metrics_path}")
print()
print("Lưu ý M05 (Hưng):")
print("  - Score method: decision_function (không phải predict_proba)")
print("  - Tất cả metrics là CV validation (không phải test)")
print("  - Test set vẫn niêm phong — Hoàng mở tại M06")